# 00 — Download
Download raw NUTNR (+ co-located METBK, for buoy instruments) streams,
annotations, and shipboard discrete (bottle) nitrate samples for one
instrument.

In [ ]:
import sys, os
import pandas as pd
import yaml
sys.path.insert(0, '..')

from ooi_data_explorations.common import get_annotations, load_kdata, list_deployments
from ooi_data_explorations.bottles import clean_data

In [ ]:
config = yaml.safe_load(open('../config/GI01SUMO-SBD11-08-NUTNRB000.yaml'))
refdes = config['refdes']
site, node, sensor = refdes.split('-', 2)
data_dir = f"../{config['paths']['data_dir']}"
os.makedirs(data_dir, exist_ok=True)
refdes

## 1. Resolve deployments
Pull the full deployment list from OOINet and drop any excluded in the config.

In [ ]:
all_deployments = sorted(list_deployments(site, node, sensor))
drop = set(config.get('deployments_to_drop', []))
deployments = [d for d in all_deployments if d not in drop]
deployments

## 2. Download NUTNR streams + annotations

In [ ]:
annotations = get_annotations(site, node, sensor)

for dN in deployments:
    dep = str(dN).zfill(4)
    for method, stream in config['streams'].items():
        load_kdata(site, node, sensor, method, stream, tag=f'deployment{dep}_{refdes}*.nc')

## 3. Download co-located METBK data (buoy instruments only)
Skipped for instruments with a co-located CTD (`met_refdes` is null).

In [ ]:
if config.get('met_refdes'):
    met_site, met_node, met_sensor = config['met_refdes'].split('-', 2)
    met_stream = config['met_streams']['recovered_inst']
    for dN in deployments:
        dep = str(dN).zfill(4)
        load_kdata(met_site, met_node, met_sensor, 'recovered_inst', met_stream,
                   tag=f'deployment{dep}_{config["met_refdes"]}*.nc')
    print(f"Downloaded METBK data from {config['met_refdes']}")
else:
    print(f'{refdes} has a co-located CTD; no METBK download needed.')

## 4. Assemble shipboard discrete (bottle) nitrate samples
Two layouts are supported (see `config/*.yaml`):
- `ship_cruise_dir` — a directory of per-cruise `Water_Sampling/*_Discrete_Summary.csv`
  files (Pioneer-NES layout), cleaned via `clean_data()` and cached to
  `data/cleaned_bottle_data.csv`.
- `bottle_csv` — an already-cleaned, per-mooring CSV (Irminger layout), used
  directly in `02-bottle-correction.ipynb`.

In [ ]:
if config.get('ship_cruise_dir'):
    bottle_data = None
    for cruise in sorted(os.listdir(config['ship_cruise_dir'])):
        cruise_path = os.path.join(config['ship_cruise_dir'], cruise, 'Water_Sampling')
        if not os.path.exists(cruise_path):
            continue
        discrete_files = [f for f in os.listdir(cruise_path) if f.endswith('Discrete_Summary.csv')]
        if not discrete_files:
            continue
        cruise_data = pd.read_csv(os.path.join(cruise_path, discrete_files[0]), index_col=None)
        bottle_data = cruise_data if bottle_data is None else pd.concat([bottle_data, cruise_data], ignore_index=True)
    bottle_data = clean_data(bottle_data)
    bottle_data.to_csv(f'{data_dir}cleaned_bottle_data.csv', index=False)
    print(f'Saved {len(bottle_data)} cleaned bottle samples')
elif config.get('bottle_csv'):
    print(f"Using pre-cleaned bottle CSV: {config['bottle_csv']}")
else:
    print('No bottle data source configured.')